# Create train, validation, and test splits

This notebook reads and merges `Data/Raw/Descriptor.csv` and `Data/Raw/Descriptor2.xlsx`, then writes split metadata into `Data/Preprocessed`.

It does not copy or move matrix `.txt` files. It validates annotated target-cell files by direct path lookup under `Data/Raw/beach_all/<date>/`; later notebooks use the saved `date` and `cell_id` values to load target cells and their same-date context from that directory.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").exists()
    or (PROJECT_ROOT / "Data").exists()
    or (PROJECT_ROOT / "pmaciag-volume-beach").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if (PROJECT_ROOT / "pmaciag-volume-beach").exists():
    PROJECT_ROOT = PROJECT_ROOT / "pmaciag-volume-beach"
if not ((PROJECT_ROOT / "Source").exists() or (PROJECT_ROOT / "Data").exists()):
    raise RuntimeError("Could not locate pmaciag-volume-beach project directory.")

DESCRIPTOR_PATHS = [
    PROJECT_ROOT / "Data" / "Raw" / "Descriptor.csv",
    PROJECT_ROOT / "Data" / "Raw" / "Descriptor2.xlsx",
]
CELL_DATA_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
PREPROCESSED_DIR = PROJECT_ROOT / "Data" / "Preprocessed"
FEATURE_SUFFIXES = ("r", "g", "b", "s", "d")

# Fractions are applied to descriptor rows. Keep their sum equal to 1.0.
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
RANDOM_SEED = 42

assert abs((TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION) - 1.0) < 1e-9

print(f"Project root:     {PROJECT_ROOT}")
print("Descriptors:")
for descriptor_path in DESCRIPTOR_PATHS:
    print(f"  - {descriptor_path}")
print(f"Cell data root:   {CELL_DATA_ROOT}")
print(f"Output directory: {PREPROCESSED_DIR}")

Project root:     /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach
Descriptors:
  - /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Raw/Descriptor.csv
  - /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Raw/Descriptor2.xlsx
Cell data root:   /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Raw/beach_all
Output directory: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed


In [3]:
DESCRIPTOR_COLUMNS = ["date", "cell_id", "class"]
DESCRIPTOR_DTYPES = {column: "string" for column in DESCRIPTOR_COLUMNS}


def read_descriptor(path):
    if not path.exists():
        raise FileNotFoundError(f"Descriptor file not found: {path}")
    if path.suffix.lower() == ".csv":
        frame = pd.read_csv(
            path,
            sep=";",
            header=None,
            names=DESCRIPTOR_COLUMNS,
            dtype=DESCRIPTOR_DTYPES,
            encoding="utf-8-sig",
        )
    elif path.suffix.lower() in {".xlsx", ".xls"}:
        frame = pd.read_excel(
            path,
            header=None,
            names=DESCRIPTOR_COLUMNS,
            dtype=DESCRIPTOR_DTYPES,
        )
    else:
        raise ValueError(f"Unsupported descriptor format: {path}")

    frame = frame.dropna(subset=DESCRIPTOR_COLUMNS).copy()
    for column in DESCRIPTOR_COLUMNS:
        frame[column] = frame[column].str.strip()
    empty_mask = frame[DESCRIPTOR_COLUMNS].eq("").any(axis=1)
    if empty_mask.any():
        raise ValueError(f"Empty descriptor values found in {path}: {int(empty_mask.sum())} rows")
    frame["source"] = path.name
    return frame


descriptor_sources = [read_descriptor(path) for path in DESCRIPTOR_PATHS]
source_counts = pd.DataFrame(
    {"source": [path.name for path in DESCRIPTOR_PATHS], "rows": [len(frame) for frame in descriptor_sources]}
)
combined_descriptor = pd.concat(descriptor_sources, ignore_index=True)

duplicate_mask = combined_descriptor.duplicated(subset=["date", "cell_id"], keep=False)
if duplicate_mask.any():
    duplicates = combined_descriptor.loc[duplicate_mask].sort_values(["date", "cell_id", "source"])
    conflicting_keys = (
        duplicates.groupby(["date", "cell_id"])["class"].nunique().loc[lambda counts: counts > 1]
    )
    if not conflicting_keys.empty:
        conflicts = duplicates.merge(
            conflicting_keys.rename("class_count"),
            on=["date", "cell_id"],
        )
        display(conflicts)
        raise ValueError("Conflicting classes found for duplicate date/cell_id pairs.")
    duplicate_key_count = duplicates[["date", "cell_id"]].drop_duplicates().shape[0]
    print(f"Dropping {len(duplicates) - duplicate_key_count:,} repeated annotations.")

descriptor = (
    combined_descriptor
    .drop_duplicates(subset=["date", "cell_id"], keep="first")
    .drop(columns="source")
    .sort_values(["date", "cell_id"])
    .reset_index(drop=True)
)
display(source_counts)
descriptor.head(), descriptor.shape

,source,rows
0,Descriptor.csv,4938
1,Descriptor2.xlsx,3184


(       date  cell_id class
 0  20180722  1000992     3
 1  20180722  1002501     7
 2  20180722  1007078     1
 3  20180722  1007106     3
 4  20180722  1011692     3,
 (8122, 3))

In [4]:
if not CELL_DATA_ROOT.exists():
    raise FileNotFoundError(f"Cell data root not found: {CELL_DATA_ROOT}")

missing_rows = []
for row in descriptor.itertuples(index=False):
    missing_features = [
        suffix
        for suffix in FEATURE_SUFFIXES
        if not (CELL_DATA_ROOT / row.date / f"{row.cell_id}{suffix}.txt").is_file()
    ]
    if missing_features:
        missing_rows.append(
            {
                "date": row.date,
                "cell_id": row.cell_id,
                "class": row[2],
                "missing_features": ",".join(missing_features),
            }
        )

missing_cells = pd.DataFrame(missing_rows)
if not missing_cells.empty:
    display(missing_cells.head(50))
    raise FileNotFoundError(
        f"{len(missing_cells):,} annotated cells are incomplete under {CELL_DATA_ROOT}."
    )

print(f"Validated five feature files for all {len(descriptor):,} annotated cells in {CELL_DATA_ROOT}.")

class_counts = descriptor["class"].value_counts().sort_index()
class_counts.to_frame(name="count")

Validated five feature files for all 8,122 annotated cells in /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Raw/beach_all.


,count
class,
0,117
1,2385
2,1447
3,665
4,706
5,645
6,536
7,856
8,723


In [5]:
train_df, holdout_df = train_test_split(
    descriptor,
    train_size=TRAIN_FRACTION,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=descriptor["class"],
)

validation_relative_fraction = VALIDATION_FRACTION / (VALIDATION_FRACTION + TEST_FRACTION)
validation_df, test_df = train_test_split(
    holdout_df,
    train_size=validation_relative_fraction,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=holdout_df["class"],
)

train_df = train_df.assign(split="train")
validation_df = validation_df.assign(split="validation")
test_df = test_df.assign(split="test")

split_df = pd.concat([train_df, validation_df, test_df], ignore_index=True)
split_df = split_df.sort_values(["split", "date", "cell_id"]).reset_index(drop=True)

split_counts = split_df["split"].value_counts().reindex(["train", "validation", "test"])
split_counts.to_frame(name="count")

,count
split,
train,5685
validation,1218
test,1219


In [6]:
expected_keys = set(map(tuple, descriptor[["date", "cell_id"]].to_numpy()))
actual_keys = set(map(tuple, split_df[["date", "cell_id"]].to_numpy()))

if expected_keys != actual_keys:
    missing = expected_keys - actual_keys
    unexpected = actual_keys - expected_keys
    raise AssertionError(f"Split mismatch. Missing={len(missing)}, unexpected={len(unexpected)}")

if split_df.duplicated(subset=["date", "cell_id"]).any():
    raise AssertionError("A date/cell_id pair appears in more than one split.")

distribution = (
    split_df
    .groupby(["split", "class"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "validation", "test"])
)

distribution

class,0,1,2,3,4,5,6,7,8,9
split,,,,,,,,,,
train,82,1669,1013,466,494,452,375,599,506,29
validation,18,358,217,99,106,96,81,128,108,7
test,17,358,217,100,106,97,80,129,109,6


In [7]:
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

columns = ["date", "cell_id", "class", "split"]
output_files = {
    "all": PREPROCESSED_DIR / "dataset_splits.csv",
    "train": PREPROCESSED_DIR / "train_split.csv",
    "validation": PREPROCESSED_DIR / "validation_split.csv",
    "test": PREPROCESSED_DIR / "test_split.csv",
}

split_df[columns].to_csv(output_files["all"], index=False)
for split_name in ["train", "validation", "test"]:
    split_df.loc[split_df["split"] == split_name, columns].to_csv(output_files[split_name], index=False)

for split_name, path in output_files.items():
    print(f"{split_name:>10}: {path}")

       all: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed/dataset_splits.csv
     train: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed/train_split.csv
validation: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed/validation_split.csv
      test: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed/test_split.csv
